# 조선시대 신하 말투 Fine-tuning (순수 HuggingFace 버전)
**모델:** Qwen2.5-1.5B-Instruct  
**방법:** QLoRA (HuggingFace + bitsandbytes)  
**환경:** Google Colab (무료 티어)

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

이 노트북은 Unsloth를 사용하지 않고, **순수 HuggingFace Transformers + bitsandbytes + PEFT**만으로 QLoRA 학습을 수행합니다.

## 1. 패키지 설치

In [ ]:
!pip install -q --upgrade transformers datasets accelerate peft trl bitsandbytes

지저분한 경고 메세지를 띄우지 않기 위한 코드

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import transformers
transformers.logging.set_verbosity_error()

## 2. 모델 로드
HuggingFace Transformers + bitsandbytes로 4bit 양자화된 모델을 불러옵니다.

### BitsAndBytesConfig 설명

| 파라미터 | 값 | 설명 |
|---------|---|------|
| `load_in_4bit` | `True` | 모델 가중치를 4bit로 양자화하여 로드 (메모리 약 4배 절약) |
| `bnb_4bit_quant_type` | `"nf4"` | NormalFloat4 양자화. 정규분포를 따르는 가중치에 최적화된 데이터 타입 |
| `bnb_4bit_compute_dtype` | `float16` | 양자화된 가중치를 연산할 때 사용하는 정밀도. float16이 T4에서 가장 빠름 |
| `bnb_4bit_use_double_quant` | `True` | 양자화 상수를 한 번 더 양자화하여 추가 메모리 절약 (약 0.4GB) |

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("✅ 모델 로드 완료")

## 3. LoRA 설정
학습할 레이어만 선택해서 메모리를 아낍니다.

### 각 파라미터 설명

#### `r = 16` (LoRA rank)
LoRA는 거대한 가중치 행렬 W를 직접 수정하지 않고, **작은 두 행렬 A, B의 곱(A×B)으로 변화량만 학습**합니다.

원래 행렬 W가 `4096 × 4096` 크기라면:
- 직접 학습: **16,777,216개** 파라미터
- LoRA (r=16): `4096×16` + `16×4096` = **131,072개** → 약 **128배 절약**

| r 값 | 용도 |
|------|------|
| 8 | 가벼운 학습, 간단한 스타일 변환 |
| 16 | 가장 흔한 선택, 대부분의 fine-tuning에 적합 |
| 32~64 | 복잡한 태스크, 더 많은 지식 주입 |

#### `lora_alpha = 16` (스케일링 계수)
LoRA가 학습한 변화량을 원래 모델에 **얼마나 세게 반영할지** 조절합니다.

실제 반영 비율 = `lora_alpha / r` = `16 / 16` = **1.0** (가장 일반적인 설정)

#### `lora_dropout = 0.05`
학습 중 LoRA 레이어의 출력을 랜덤하게 꺼서 과적합을 방지하는 기법입니다. 순수 HuggingFace 버전에서는 Unsloth의 최적화 커널이 없으므로, 소량의 dropout(0.05)을 적용하여 과적합을 방지합니다.

#### `bias = "none"`
모델의 bias 파라미터를 학습하지 않습니다. 대부분의 경우 bias를 학습하지 않아도 성능 차이가 거의 없어서 `"none"`이 표준입니다.

#### `task_type = "CAUSAL_LM"`
이 LoRA가 적용될 모델의 태스크 유형입니다. Causal Language Model(GPT 계열의 자기회귀 모델)임을 명시합니다.

#### `target_modules`
LoRA를 적용할 레이어를 지정합니다. Transformer의 Attention과 FFN 레이어에 적용하여 효율적으로 학습합니다.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()

model.print_trainable_parameters()
print("✅ LoRA 설정 완료")

## 4. 데이터 준비
`joseon_finetune_200.json`을 업로드한 뒤 실행하세요.

In [ ]:
# 파일 업로드
from google.colab import files
uploaded = files.upload()  # joseon_finetune_200.json 선택

In [ ]:
import json
from datasets import Dataset

with open("joseon_finetune_200.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# Qwen 채팅 템플릿 적용
def format_example(example):
    messages = [
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw).map(format_example)
print(f"✅ 데이터셋 준비 완료: {len(dataset)}개")
print("\n--- 샘플 확인 ---")
print(dataset[0]["text"])

## 5. 학습 실행
약 5~10분 소요됩니다.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        max_length=1024,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        fp16=False,
        bf16=False,
        logging_steps=10,
        output_dir="./joseon_output",
        report_to="none",
        gradient_checkpointing_kwargs={"use_reentrant": False},
    ),
)

trainer.train()
print("✅ 학습 완료")

## 6. 결과 비교
fine-tuning 전/후 말투를 비교합니다.

In [ ]:
def generate(prompt, model, max_new_tokens=200):
    model.eval()
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=False,
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
        )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# 테스트할 질문
test_questions = [
    "오늘 점심 뭐 먹을까?",
    "요즘 너무 피곤해.",
    "주말에 어디 가면 좋을까?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {generate(q, model)}")
    print("-" * 50)

## 7. 모델 저장 (선택)
LoRA 가중치만 저장합니다 (용량 작음).

In [ ]:
model.save_pretrained("joseon_lora")
tokenizer.save_pretrained("joseon_lora")

# 구글 드라이브에 저장하려면
# from google.colab import drive
# drive.mount('/content/drive')
# model.save_pretrained("/content/drive/MyDrive/joseon_lora")

print("✅ 저장 완료 → joseon_lora/")

## 8. 모델 불러오기

모델을 불러온 뒤, LoRA 가중치를 따로 불러와서 적용합니다.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 1) 베이스 모델 로드 (학습할 때와 동일한 설정)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2) 저장해둔 LoRA 가중치 얹기
model = PeftModel.from_pretrained(model, "joseon_lora")
print("✅ 모델 불러오기 완료")